**Датасет:** результаты ab теста конверсии в покупку после внедрения нового дизайна сайта

---


**Задача:** определить целесообразность внедрения нового дизайна сайта

---


**H0:** новый дизайн на конверсию не влияет, разница между контрольной выборкой и тестовой незначима


> Уровень значимости alpha = 0.05 — стандартный порог для продуктовых A/B тестов


---



**H1:** новый дизайн сайта изменил конверсию, пользователям удобнее совершать целевое действие, поэтому конверсия изменилась в любую сторону

In [ ]:
import pandas as pd
import numpy as np

df = pd.read_csv('ab_data.csv')

print('Размер датасета:', df.shape)
print('\nПервые 5 строк:')
df.head()

Размер датасета: (294478, 5)

Первые 5 строк:


,user_id,timestamp,group,landing_page,converted
0,851104,2017-01-21 22:11:48.556739,control,old_page,0
1,804228,2017-01-12 08:01:45.159739,control,old_page,0
2,661590,2017-01-11 16:55:06.154213,treatment,new_page,0
3,853541,2017-01-08 18:28:03.143765,treatment,new_page,0
4,864975,2017-01-21 01:52:26.210827,control,old_page,1


**Этап 1. Очистка данных.**

---


Загружаем датасет ab_data.csv — 294478 строк, 5 столбцов: user_id, timestamp, group, landing_page, converted



Проблемы: 3894 пользователя встречались в датасете более одного раза.
1006 + 1038 значения group с ошибочным определением landing_page.

Решение: оставить в выборке значение
landing_page, являющееся первым; оставить только group с верным значением landing_page

In [ ]:
df.isnull().sum()

,0
user_id,0
timestamp,0
group,0
landing_page,0
converted,0


In [ ]:
df['user_id'].nunique()

290584

In [ ]:
df.shape[0]

294478

In [ ]:
df['user_id'].value_counts()[df['user_id'].value_counts() > 1]

,count
user_id,
752737,2
781280,2
767913,2
886060,2
731779,2
...,...
835630,2
931722,2
808992,2


In [ ]:
df_clean = df.drop_duplicates(subset='user_id', keep='first')
df_clean.shape

(290584, 5)

In [ ]:
df_clean['group'].value_counts()

,count
group,
treatment,145352
control,145232


In [ ]:
df_clean[(df_clean['group'] == 'control') & (df_clean['landing_page'] == 'new_page')]

,user_id,timestamp,group,landing_page,converted
22,767017,2017-01-12 22:58:14.991443,control,new_page,0
240,733976,2017-01-11 15:11:16.407599,control,new_page,0
490,808613,2017-01-10 21:44:01.292755,control,new_page,0
846,637639,2017-01-11 23:09:52.682329,control,new_page,1
850,793580,2017-01-08 03:25:33.723712,control,new_page,1
...,...,...,...,...,...
277083,731502,2017-01-16 05:54:36.502340,control,new_page,0
277283,904541,2017-01-24 12:19:25.218304,control,new_page,0
281176,828478,2017-01-13 23:45:27.822482,control,new_page,0
281593,917949,2017-01-15 03:16:02.243648,control,new_page,1


In [ ]:
df_clean[(df_clean['group'] == 'treatment') & (df_clean['landing_page'] == 'old_page')]

,user_id,timestamp,group,landing_page,converted
308,857184,2017-01-20 07:34:59.832626,treatment,old_page,0
327,686623,2017-01-09 14:26:40.734775,treatment,old_page,0
357,856078,2017-01-12 12:29:30.354835,treatment,old_page,0
685,666385,2017-01-23 08:11:54.823806,treatment,old_page,0
713,748761,2017-01-10 15:47:44.445196,treatment,old_page,0
...,...,...,...,...,...
274590,879231,2017-01-03 12:08:24.976156,treatment,old_page,0
282645,717723,2017-01-10 18:43:58.148113,treatment,old_page,0
285987,914482,2017-01-05 16:55:40.060421,treatment,old_page,0
286353,767924,2017-01-08 21:48:53.436050,treatment,old_page,0


In [ ]:
df_clean = df_clean[((df_clean['group'] == 'control') & (df_clean['landing_page'] == 'old_page')) |
    ((df_clean['group'] == 'treatment') & (df_clean['landing_page'] == 'new_page'))]

In [ ]:
df_clean.shape

(288540, 5)

После очистки в группе control 144226 значений, в treatment 144314 значений. Конверсия составила 12,03% и 11,87% соответственно. Предварительное наблюдение: в тестовой выборке конверсия снизилась на 0,16 процентных пунктов

In [ ]:
conversion = df_clean.groupby('group').agg({'converted': 'sum', 'user_id': 'count'})
conversion

,converted,user_id
group,,
control,17349,144226
treatment,17134,144314


In [ ]:
conversion['conversion_rate'] = conversion['converted'] / conversion['user_id']
conversion

,converted,user_id,conversion_rate
group,,,
control,17349,144226,0.120290
treatment,17134,144314,0.118727


статистическую значимость полученных результатов, а именно не являются ли они случайным значением или шумом проверим с помощью z-теста. в ходе исследования было получено значение p-value = 0.195, что выше порога 0.05, вследствие чего, гипотезу H0 мы не отвергаем.

In [ ]:
from statsmodels.stats.proportion import proportions_ztest

In [24]:
conversions = conversion['converted'].values
users = conversion['user_id'].values
conversions, users


(array([17349, 17134]), array([144226, 144314]))

In [26]:
stat, p_value = proportions_ztest(conversions, users)
print('p_value:', p_value)

p_value: 0.19558364287881436


Мы видим что конверсия в тестовой группе составила 11.87% против 12.03% в контрольной — разница 0.16 процентных пункта. p-value = 0.195, что выше порога 0.05. Это означает что нет статистически значимых доказательств влияния нового дизайна на конверсию. Рекомендую: 1) убедиться что тест был завершён в запланированный срок а не остановлен раньше, 2) рассчитать необходимый размер выборки через power analysis перед запуском следующего теста, 3) провести повторный тест с правильно рассчитанной выборкой.

